# 01 — Generate the synthetic light-curve set

Builds a synthetic copy of `light_curves/` with known generating parameters, plus the
ground-truth table.

Each real FITS is copied and **only the `flux` column is replaced**, so timestamps,
`flux_err`, per-sector HDU structure and headers are inherited unchanged. The output
tree mirrors `light_curves/` exactly (`MW`/`LMC`/`SMC` subdirs, same filenames), which
is what lets the pipeline run on it with no code changes — only `LC_DIR` differs.

Runtime is about a minute for all 348 clusters.


In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
import numpy as np
import pandas as pd

# GenerateTable_CGW must be importable so table_pipeline / gp_pipeline resolve.
MODULE_DIR = os.path.abspath('..')
if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

DATA_DIR      = '../../data'                 # Wainer catalogues
REAL_LC_DIR   = '../../light_curves'         # source of structure, times, errors
OUT_LC_DIR    = '../../light_curves_synthetic'
RN_TABLE      = '../data/cluster_table_3sectorcombos_per_cluster_SHOQprior_RNperiodCut.pkl'
TRUTH_PATH    = 'data/ground_truth.pkl'
SEED          = 0

print('OUT_LC_DIR :', os.path.abspath(OUT_LC_DIR))
print('TRUTH_PATH :', os.path.abspath(TRUTH_PATH))

OUT_LC_DIR : /astro/users/cgwill/TESS_Cluster_Age_ML/light_curves_synthetic
TRUTH_PATH : /astro/users/cgwill/TESS_Cluster_Age_ML/GenerateTable_CGW/sytheticdata_generate_compare/data/ground_truth.pkl


## Generative model

```
x(t) = 1 + red(t; alpha, rms) + sum_i A_i sin(2 pi t / P_i + phi_i)
```

| parameter | distribution | note |
|---|---|---|
| `N` oscillators | `Uniform{0..10}` | `N=0` is a pure-red-noise null class |
| `A_i` | `log10 A ~ U(-5, -3)` | calibrated so total RMS matches the real 3.2e-4 |
| `P_i` | `Uniform(0.1, 9)` d | linear, entirely inside the 0.083–10 d search band |
| `(alpha, log10_N)` | resampled **jointly** from the real table | preserves their +0.87 correlation |

The red-noise amplitude is *calibrated*, not assumed: `rn_log10_N` is a fit output in
periodogram units, so the generator measures what its own realisation produces on the
reference sector and rescales (power ∝ amplitude², so one measurement is exact).

In [2]:
from sytheticdata_generate_compare.synth.run_generate import run
from sytheticdata_generate_compare.synth import generate as G

print('amplitude range  :', G.LOG10_AMP_RANGE)
print('period range (d) :', G.PERIOD_RANGE)
print('max oscillators  :', G.N_OSC_MAX)

amplitude range  : (-5.0, -3.0)
period range (d) : (0.1, 9.0)
max oscillators  : 10


In [3]:
%%time
truth = run(data_dir=DATA_DIR, real_lc_dir=REAL_LC_DIR, out_lc_dir=OUT_LC_DIR,
            rn_table_path=RN_TABLE, out_truth_path=TRUTH_PATH,
            seed=SEED, limit=None, verbose=True)
truth.head()

  MW: 124 clusters from Wainer2023_Table1_MW.txt
  SMC: 106 clusters from Wainer2023_Table2_SMC.txt
  LMC: 118 clusters from Wainer2023_Table3_LMC.txt
[build_base_table] total: 348 clusters
  [25/348] ok=25 failed=0
  [50/348] ok=50 failed=0
  [75/348] ok=75 failed=0
  [100/348] ok=100 failed=0
  [125/348] ok=125 failed=0
  [150/348] ok=150 failed=0
  [175/348] ok=175 failed=0
  [200/348] ok=200 failed=0
  [225/348] ok=225 failed=0
  [250/348] ok=250 failed=0
  [275/348] ok=275 failed=0
  [300/348] ok=300 failed=0
  [325/348] ok=325 failed=0
[run_generate] wrote 348 rows -> data/ground_truth.pkl
[run_generate] done | ok=348 failed=0
CPU times: user 31min 44s, sys: 871 ms, total: 31min 45s
Wall time: 42.3 s


,age,name,origin,path,rn_rms_realized,err_frac_med,n_osc,periods,amplitudes,phases,...,true_intrinsic_std,true_vn_ratio,true_rms,true_n_points,true_baseline_days,true_stetson_j,inj_var_sinusoids,inj_amp_max,inj_period_of_max,inj_amp_sum
0,7.95,ASCC 116,MW,../../light_curves_synthetic/MW/hlsp_elk_tess_...,0.000319,0.000061,9,"[2.501101752498446, 0.46466436303213277, 0.247...","[0.00042825960119971, 1.0126911166161174e-05, ...","[2.6558221377616955, 0.17793774164533058, 0.78...",...,0.000723,13.507409,0.000723,54886,1143.437500,9.103986,4.179078e-07,0.000533,6.592519,0.001972
1,9.26,ASCC 57,MW,../../light_curves_synthetic/MW/hlsp_elk_tess_...,0.000022,0.000032,7,"[1.1150217750958589, 2.413016654766122, 1.5567...","[2.573163910967085e-05, 0.00012872910142748008...","[3.5181548554463298, 3.6897670705445016, 1.817...",...,0.000547,140.847647,0.000547,36661,763.750000,13.923910,2.978674e-07,0.000498,7.603182,0.001642
2,7.77,ASCC 8,MW,../../light_curves_synthetic/MW/hlsp_elk_tess_...,0.000018,0.000045,2,"[8.75209451793066, 0.5716773139597501]","[1.4414956452800046e-05, 0.0001449473790950666]","[4.603040575620673, 3.582747206676822]",...,0.000104,11.130602,0.000104,53731,1119.375000,1.964292,1.060877e-08,0.000145,0.571677,0.000159
3,8.75,ASCC 81,MW,../../light_curves_synthetic/MW/hlsp_elk_tess_...,0.000056,0.000039,6,"[3.6193256035205867, 2.3488623339791483, 6.350...","[1.3127333015181974e-05, 9.164545039485083e-05...","[0.9666431476770064, 0.1969477250853051, 0.605...",...,0.000349,48.683564,0.000349,36709,764.750000,7.503278,1.189442e-07,0.000433,6.350524,0.000798
4,7.00,ASCC 9,MW,../../light_curves_synthetic/MW/hlsp_elk_tess_...,0.000030,0.000019,3,"[1.4768507333290384, 1.0101401447810339, 2.969...","[4.466334991895529e-05, 4.1493294451113154e-05...","[4.9567375242862, 0.08688827915673221, 4.17784...",...,0.000051,3.973306,0.000051,1170,24.354167,1.832133,2.001281e-09,0.000045,1.476851,0.000103


## Check the layout resolves

This is the check that the pipeline will run unmodified: `get_lc_path` must find every
cluster in the synthetic tree, with the same count as the real one.

In [4]:
from table_pipeline.wainer_table import build_base_table
from table_pipeline.lc_lsp import get_lc_path

base = build_base_table(DATA_DIR)
ok = miss = 0
for r in base.itertuples(index=False):
    try:
        get_lc_path(r.name, r.origin, OUT_LC_DIR); ok += 1
    except FileNotFoundError:
        miss += 1
        print('  MISSING:', r.name, r.origin)
print(f'resolved {ok}/{len(base)}   missing={miss}   (expect {len(base)} / 0)')

  MW: 124 clusters from Wainer2023_Table1_MW.txt
  SMC: 106 clusters from Wainer2023_Table2_SMC.txt
  LMC: 118 clusters from Wainer2023_Table3_LMC.txt
[build_base_table] total: 348 clusters
resolved 348/348   missing=0   (expect 348 / 0)


## Sanity: injected vs realised variability

In [5]:
truth['var_red'] = truth['true_excess_var'] - truth['inj_var_sinusoids']
print('total intrinsic RMS  median = %.2e   (real: 3.2e-4)'
      % np.sqrt(truth['true_excess_var']).median())
print('red-noise RMS        median = %.2e' % np.sqrt(truth['var_red'].clip(0)).median())
print('sinusoid RMS         median = %.2e' % np.sqrt(truth['inj_var_sinusoids']).median())
print()
print(truth['n_osc'].value_counts().sort_index().to_string())

total intrinsic RMS  median = 7.29e-04   (real: 3.2e-4)
red-noise RMS        median = 4.51e-04
sinusoid RMS         median = 4.20e-04

n_osc
0     45
1     28
2     30
3     28
4     26
5     30
6     38
7     28
8     28
9     28
10    39


## Next

Point `generate_table.ipynb` at the synthetic directory and run the pipeline as usual:

```python
LC_DIR = '../light_curves_synthetic'
OUTPUT = './data/cluster_table_synthetic'
```

Then run `02_compare_to_truth.ipynb`.